In [3]:
import sys
import os.path
!pip install uproot awkward vector matplotlib numpy pandas lmfit

  Using cached uproot-5.7.4-py3-none-any.whl.metadata (35 kB)
  Using cached awkward-2.9.1-py3-none-any.whl.metadata (7.4 kB)
  Using cached vector-1.8.1-py3-none-any.whl.metadata (15 kB)
  Using cached matplotlib-3.11.0-cp311-cp311-macosx_10_12_x86_64.whl.metadata (80 kB)
  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.3-cp311-cp311-macosx_10_9_x86_64.whl.metadata (79 kB)
  Using cached lmfit-1.3.4-py3-none-any.whl.metadata (8.8 kB)
  Using cached cramjam-2.11.0-cp311-cp311-macosx_10_12_x86_64.whl.metadata (5.6 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.8.0-cp311-cp311-macosx_10_9_x86_64.whl.metadata (14 kB)
  Using cached awkward_cpp-53-cp311-cp311-macosx_10_9_x86_64.whl.metadata (2.1 kB)
  Using cached importlib_metadata-9.0.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_10_9_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-

In [4]:
import numpy as np # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
import matplotlib_inline # to edit the inline plot format
#matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg') # to make plots in pdf (vector) format
from matplotlib.ticker import AutoMinorLocator # for minor ticks
import uproot # for reading .root files
import awkward as ak # to represent nested data in columnar format
import vector # for 4-momentum calculations
import time # for printing time stamps
import requests # for file gathering, if needed

Matplotlib is building the font cache; this may take a moment.


In [6]:
MeV = 0.001
GeV = 1.0
import atlasopenmagic as atom
atom.available_releases()
atom.set_release('2025e-13tev-beta')

Fetching metadata for release: 2025e-13tev-beta...


Available releases:
2016e-8tev           2016 Open Data for education release of 8 TeV proton-proton collisions (https://opendata.cern.ch/record/3860).
2020e-13tev          2020 Open Data for education release of 13 TeV proton-proton collisions (https://cern.ch/2r7xt).
2024r-pp             2024 Open Data for research release for proton-proton collisions (https://opendata.cern.record/80020).
2024r-hi             2024 Open Data for research release for heavy-ion collisions (https://opendata.cern.ch/record/80035).
2025e-13tev-beta     2025 Open Data for education and outreach beta release for 13 TeV proton-proton collisions (https://opendata.cern.ch/record/93910).
2025r-evgen-13tev    2025 Open Data for research release for event generation at 13 TeV (https://opendata.cern.ch/record/160000).
2025r-evgen-13p6tev  2025 Open Data for research release for event generation at 13.6 TeV (https://opendata.cern.ch/record/160000).


Fetching datasets: 100%|██████████| 374/374 [00:00<00:00, 1080.05datasets/s]
✓ Successfully cached 374 datasets.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


In [7]:
lumi = 36.6 #fb-1
fraction = 0.05 #pick how fast code runs
skim = "exactly4lep"

In [8]:
defs = {
    r'Data':{'dids':['data']},
    r'Background $Z,t\bar{t},t\bar{t}+V,VVV$':{'dids': [410470,410155,410218,
                                                        410219,412043,364243,
                                                        364242,364246,364248,
                                                        700320,700321,700322,
                                                        700323,700324,700325], 'color': "#6b59d3" }, # purple
    r'Background $ZZ^{*}$':     {'dids': [700600],'color': "#ff0000" },# red
    r'Signal ($m_H$ = 125 GeV)':  {'dids': [345060, 346228, 346310, 346311, 346312,
                                          346340, 346341, 346342],'color': "#00cdff" },# light blue
}

samples   = atom.build_dataset(defs, skim=skim, protocol='https', cache=True)

In [12]:
data15_periodD = samples['Data']['list'][0]
print(f"{data15_periodD = }")

data15_periodD = 'simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_exactly4lep_data15_periodD.exactly4lep.root'


In [13]:
!pip install requests aiohttp
#tree = uproot.open(data15_periodD + ":analysis")

  Using cached aiohttp-3.14.1-cp311-cp311-macosx_10_9_x86_64.whl.metadata (8.3 kB)
  Using cached aiohappyeyeballs-2.6.2-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-macosx_10_9_x86_64.whl.metadata (20 kB)
  Using cached multidict-6.7.1-cp311-cp311-macosx_10_9_x86_64.whl.metadata (5.3 kB)
  Using cached propcache-0.5.2-cp311-cp311-macosx_10_9_x86_64.whl.metadata (16 kB)
  Using cached yarl-1.24.2-cp311-cp311-macosx_10_9_x86_64.whl.metadata (94 kB)
Using cached aiohttp-3.14.1-cp311-cp311-macosx_10_9_x86_64.whl (518 kB)
Using cached multidict-6.7.1-cp311-cp311-macosx_10_9_x86_64.whl (44 kB)
Using cached yarl-1.24.2-cp311-cp311-macosx_10_9_x86_64.whl (91 kB)
Using cached aiohappyeyeballs-2.6.2-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using 

In [15]:
tree = uproot.open(data15_periodD + ":analysis")
print(tree.num_entries)
print(tree.keys())
print(tree.arrays())

3424
['num_events', 'sum_of_weights', 'sum_of_weights_squared', 'category', 'TriggerMatch_DILEPTON', 'ScaleFactor_MLTRIGGER', 'ScaleFactor_PILEUP', 'ScaleFactor_FTAG', 'mcWeight', 'xsec', 'filteff', 'kfac', 'channelNumber', 'eventNumber', 'runNumber', 'trigML', 'trigP', 'trigDT', 'trigT', 'trigE', 'trigDM', 'trigDE', 'trigM', 'trigMET', 'ScaleFactor_BTAG', 'ScaleFactor_JVT', 'jet_n', 'jet_pt', 'jet_eta', 'jet_phi', 'jet_e', 'jet_btag_quantile', 'jet_jvt', 'largeRJet_n', 'largeRJet_pt', 'largeRJet_eta', 'largeRJet_phi', 'largeRJet_e', 'largeRJet_m', 'largeRJet_D2', 'jet_pt_jer1', 'jet_pt_jer2', 'ScaleFactor_ELE', 'ScaleFactor_MUON', 'ScaleFactor_LepTRIGGER', 'ScaleFactor_MuTRIGGER', 'ScaleFactor_ElTRIGGER', 'lep_n', 'lep_type', 'lep_pt', 'lep_eta', 'lep_phi', 'lep_e', 'lep_charge', 'lep_ptvarcone30', 'lep_topoetcone20', 'lep_z0', 'lep_d0', 'lep_d0sig', 'lep_isTightID', 'lep_isMediumID', 'lep_isLooseID', 'lep_isTightIso', 'lep_isLooseIso', 'lep_isTrigMatched', 'ScaleFactor_PHOTON', 'phot

In [18]:
#examine lepton energies
#tree["lep_e"].arrays(library="ak")

In [17]:
# define which variables are important to our analysis
variables = ['lep_pt','lep_eta','lep_phi','lep_e','lep_charge','lep_type','trigE','trigM','lep_isTrigMatched',
            'lep_isLooseID','lep_isMediumID','lep_isLooseIso','lep_type']
for data in tree.iterate(variables,library="ak"):
    print(data)

[{lep_pt: [32.4, 11.4, 11.6, 8.2], lep_eta: [-1.7, ...], ...}, ..., {...}]


In [ ]:
# do cuts for first event
entry = tree.arrays(library="ak")[:1] # first entry of the tree

# cut on lepton type 
# e- is 11, mu- is 13
lep_type = entry['lep_type']
print(lep_type)
sum_lep_type = lep_type[:,0]+lep_type[:,1]+lep_type[:, 2] + lep_type[:, 3]
print(sum_lep_type)
# we need two pairs of leptons, can either have 4e -> 4*11=44, 4e+2mu-> 22+26=48, 4mu -> 52
# otherwise should be cut
lep_type_cut_bool = (sum_lep_type != 44) & (sum_lep_type != 48) & (sum_lep_type != 52)
print(f"Cut for lepton type?: {lep_type_cut_bool}") #True means we cut

# cut on lepton charge
# first lepton in each event is [:,0], 2nd lepton is [:,1], etc.
lep_charge = entry['lep_charge']
sum_lep_charge = lep_charge[:,0] + lep_charge[:,1] + lep_charge[:,2] + lep_charge[:,3]
lep_charge_cut_bool = (sum_lep_charge != 0)
print(f"Cut for lepton charge?: {lep_charge_cut_bool}")

# calculate invariant mass of the 4-lep state
p4 = vector.zip({"pt":entry['lep_pt'], "eta": entry['lep_eta'], "phi": entry['lep_phi'], "E": entry['lep_e']})
invar_mass = (p4[:,0]+p4[:,1] + p4[:,2] + p4[:,3]).M
print(p4[:,0])
print(f"Invariant mass: {invar_mass}")

[[11, 11, 11, 13]]
[46]
Cut for lepton type?: [True]
Cut for lepton charge?: [False]
[{rho: 32.4, phi: 1.7, eta: -1.7, t: 91.3}]
Invariant mass: [5.6]


In [ ]:
# write functions for cuts
def cut_lep_type(lep_type):
    sum_lep_type = lep_type[:,0]+lep_type[:,1]+lep_type[:, 2] + lep_type[:, 3]
    lep_type_cut_bool = (sum_lep_type != 44) & (sum_lep_type != 48) & (sum_lep_type != 52)
    print(f"Cut for lepton type?: {lep_type_cut_bool}") #True means we cut
    return lep_type_cut_bool

def cut_lep_charge(lep_charge):
    lep_charge = entry['lep_charge']
    sum_lep_charge = lep_charge[:,0] + lep_charge[:,1] + lep_charge[:,2] + lep_charge[:,3]
    lep_charge_cut_bool = (sum_lep_charge != 0)
    print(f"Cut for lepton charge?: {lep_charge_cut_bool}")
    return lep_charge_cut_bool

def calc_mass(lep_pt,lep_eta,lep_phi,lep_e):
    p4 = vector.zip({"pt": lep_pt, "eta": lep_eta, "phi": lep_phi, "E": lep_e})
    invar_mass = (p4[:,0]+p4[:,1] + p4[:,2] + p4[:,3]).M
    print(f"Invariant mass: {invar_mass}")
    return invar_mass

# this catches if the lepton that the event was triggered on matches one of the 4 leptons we care about
def cut_trig_match(lep_trigmatch):
    trigmatch = lep_trigmatch
    cut1 = ak.sum(trigmatch,axis=1) >= 1
    return cut1

# see if we trigger on either muon or electron
def cut_trig(trigE,trigM):
    return trigE | trigM

# check if all four lepton matches isolation and identification criteria
# these criteria ensure that the lepton signature isn't a background from a jet or other process
def ID_iso_cut(IDel,IDmu,isoel,isomu,pid):
    thispid = pid
    return (ak.sum(((thispid==13) & IDmu & isomu) | (thispid==11 & IDel &isoel), axis=1) == 4)